# Decoder-only LLM：逐块运行（LLaMA 风格）

这个笔记本把 [decoder_only_llm.py](./decoder_only_llm.py) 拆成小块，从 token id 一路走到 loss 和带 KV cache 的生成。结构和现在的开源大模型（LLaMA、Qwen、Mistral）一致，只是尺寸小到能把每个张量打印出来看。

和 [transformer_encoder_decoder.ipynb](./transformer_encoder_decoder.ipynb) 相比，这里只有一条因果的 token 流：没有编码器，没有交叉注意力，prompt 和续写是同一个序列。剩下的部分换成了现代模型的做法：

| 部件 | 2017 Transformer / GPT-2 | 这个笔记本（LLaMA 风格） |
|---|---|---|
| 位置 | 输入处加位置向量 $P$ | 不加；在注意力里用 RoPE 旋转 $Q$、$K$ |
| 归一化 | LayerNorm（2017 post-norm，GPT-2 pre-norm） | RMSNorm，pre-norm |
| 注意力 | MHA：每个头一组 $K$、$V$ | GQA：几个 query 头共用一组 $K$、$V$ |
| FFN | ReLU 或 GELU，宽 $4d$ | SwiGLU，宽约 $\tfrac{8}{3}d$ |
| 偏置 | 每个线性层都有 | 全部没有（LLaMA、Mistral、Qwen3；Qwen2/2.5 在 $W^Q$、$W^K$、$W^V$ 上保留偏置） |
| 生成（推理写法，不是结构差异） | 那个笔记本的 `greedy_decode` 为了简单每步重算整个前缀（GPT-2 的官方代码其实已经用 KV cache） | prefill 一次，之后每步只算新 token，读 KV cache |

每一步都分三段：**数学**（公式、输入和输出的形状）、**代码**（和 `.py` 一字不差）、**运行**（在例子上跑，用 `assert` 核对代码和公式一致）。

**怎么运行。** 在 VS Code 里选 `.venv-transformer` 作为 kernel，从上到下运行。测试：`.venv-transformer/bin/python -m unittest -v test_decoder_only_llm`。

## 0. 导入、配置和例子

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Optional

import torch
from torch import Tensor, nn
from torch.nn import functional as F

torch.manual_seed(0)
print("torch", torch.__version__)

torch 2.8.0


### 0.1 配置

所有尺寸放在一个 `Config` 里。右边一列是 LLaMA 3 8B 的真实值，可以对照着看每个数在大模型里有多大。

| 字段 | 含义 | 这里 | LLaMA 3 8B |
|---|---|---|---|
| `vocab_size` | 词表大小 $V$ | 11 | 128,256 |
| `d_model` | 每个 token 向量的宽度 $d$ | 16 | 4,096 |
| `n_layers` | block 的层数 $L$ | 2 | 32 |
| `n_heads` | query 头数 $h$ | 4 | 32 |
| `n_kv_heads` | key/value 头数 $h_{kv}$ | 2 | 8 |
| `head_dim` | 每头宽度 $d_h = d / h$ | 4 | 128 |
| `d_ff` | SwiGLU 中间宽度 | 48 | 14,336 |
| `max_seq_len` | RoPE 表的行数（最长序列） | 16 | 8,192 |
| `rope_base` | RoPE 的底数 | 10,000 | 500,000 |
| `tie_embeddings` | 输入和输出是否共用一个矩阵 | 否 | 否 |

- **$h_{kv} = h$ 就是普通的 MHA，$h_{kv} = 1$ 就是 MQA**，介于两者之间就是 GQA。这里 4 个 query 头分成 2 组，每组共用一个 $K$、$V$ 头。
- **`d_ff = 48`**：SwiGLU 有三个矩阵，所以中间宽度取约 $\tfrac{8}{3}d = 42.7$，再向上取到 8 的倍数，让参数量和 $4d$ 的普通 FFN 差不多。LLaMA 3 8B 取的是 $3.5d$。

In [2]:
@dataclass
class Config:
    vocab_size: int = 11
    d_model: int = 16          # d: width of every token vector
    n_layers: int = 2          # L: number of blocks
    n_heads: int = 4           # h: query heads
    n_kv_heads: int = 2        # h_kv: key/value heads; h_kv = h is MHA, h_kv = 1 is MQA
    d_ff: int = 48             # SwiGLU width: 8/3 d = 42.7, rounded up to a multiple of 8
    max_seq_len: int = 16      # rows of the RoPE tables
    rope_base: float = 10000.0
    norm_eps: float = 1e-6
    tie_embeddings: bool = False

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads

In [3]:
cfg = Config()
print(cfg)
print("head_dim =", cfg.head_dim, "  query heads per K/V head =", cfg.n_heads // cfg.n_kv_heads)

Config(vocab_size=11, d_model=16, n_layers=2, n_heads=4, n_kv_heads=2, d_ff=48, max_seq_len=16, rope_base=10000.0, norm_eps=1e-06, tie_embeddings=False)
head_dim = 4   query heads per K/V head = 2


### 0.2 例子：两句话

训练数据只有两句话，开头放 `<bos>`，结尾放 `<eos>`：

$$
\begin{aligned}
&\texttt{<bos> we watch a short video <eos>} \\
&\texttt{<bos> you read a short book <eos>}
\end{aligned}
$$

两句的中间都是 `a short`，最后一个词是 `video` 还是 `book`，取决于四个位置之前的第一个词。模型必须用注意力回头看，才能答对。

decoder-only 模型只有一个序列：输入是它，标签也是它，只是错开一格。第一句的 7 个位置：

| 位置 $j$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| 输入 $t_j$ | `<bos>` | we | watch | a | short | video | `<eos>` |
| 标签 $t_{j+1}$ | we | watch | a | short | video | `<eos>` | （无） |

最后一个位置没有标签，所以 7 个 token 只产生 6 个预测。

In [4]:
VOCAB = ["<pad>", "<bos>", "<eos>", "we", "watch", "a", "short", "video", "you", "read", "book"]
PAD, BOS, EOS = 0, 1, 2
SENTENCES = [["we", "watch", "a", "short", "video"], ["you", "read", "a", "short", "book"]]


def encode(words: list) -> Tensor:
    return torch.tensor([[BOS] + [VOCAB.index(w) for w in words] + [EOS]])


def decode_ids(ids: Tensor) -> str:
    return " ".join(VOCAB[i] for i in ids.tolist())

In [5]:
ids = encode(SENTENCES[0])
print("ids", list(ids.shape), ids[0].tolist(), "=", decode_ids(ids[0]))
B, n = ids.shape

ids [1, 7] [1, 3, 4, 5, 6, 7, 2] = <bos> we watch a short video <eos>


## 1. 第 1 步：token → embedding，不加位置

**输入** ids `[B, n]`，**输出** $X$ `[B, n, d]`。

$$
X = E[\mathbf{t}], \qquad E \in \mathbb{R}^{V \times d}
$$

就是按行号查表。和 2017 Transformer、GPT-2 不同，这里**不加位置向量**；也不像 2017 Transformer 那样把 embedding 乘 $\sqrt{d}$（GPT-2 本来就不乘）。位置信息在第 3 步通过 RoPE 进入注意力。模型里这张表是 `DecoderLM.embed`，先单独建一张看看：

In [6]:
embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
X = embed(ids)
print("ids", list(ids.shape), "-> X", list(X.shape))
assert torch.equal(X[0, 1], embed.weight[VOCAB.index("we")])   # row 1 of X is the table row of "we"

ids [1, 7] -> X [1, 7, 16]


## 2. 第 2 步：RMSNorm

**输入**和**输出**都是 `[B, n, d]`，每个 token 单独处理。

$$
\operatorname{RMSNorm}(x) = \frac{x}{\sqrt{\tfrac{1}{d}\sum_{k} x_k^2 + \epsilon}} \odot g
$$

和 LayerNorm 比，它**不减均值，也没有 $\beta$**，只把每个 token 向量的均方根缩放到 1，再乘一个可学习的 $g$。计算更省，效果相当，所以现在的大模型基本都用它。实现里先转成 float32 再算，和 LLaMA 一样，避免半精度下平方溢出。

它不是只在 embedding 后用一次：pre-norm 把它放在每个子层的入口，每个 block 里注意力和 FFN 之前各一个，输出前再一个（第 6、7 步）。下面先在 $X$ 上单独跑一次，得到的就是第 4 步注意力的输入。

In [7]:
class RMSNorm(nn.Module):
    """RMSNorm(x) = x / sqrt(mean(x^2) + eps) * g: rescale each token, no mean subtraction, no bias."""

    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))   # g, learned, one per feature

    def forward(self, x: Tensor) -> Tensor:
        x32 = x.float()                               # like LLaMA, compute the norm in float32 even for fp16/bf16 inputs
        return (x32 * torch.rsqrt(x32.pow(2).mean(dim=-1, keepdim=True) + self.eps)).type_as(x) * self.weight

In [8]:
norm = RMSNorm(cfg.d_model)
Xn = norm(X)
rms = Xn.pow(2).mean(dim=-1).sqrt()
print("RMS of each token after RMSNorm:", [round(v, 3) for v in rms[0].tolist()])   # all about 1
manual = X / torch.sqrt(X.pow(2).mean(-1, keepdim=True) + norm.eps) * norm.weight
assert torch.allclose(Xn, manual, atol=1e-6)
print("mean of each token is NOT moved to 0:", [round(v, 2) for v in Xn.mean(dim=-1)[0].tolist()])

RMS of each token after RMSNorm: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
mean of each token is NOT moved to 0: [0.12, 0.23, 0.07, 0.03, -0.15, 0.1, 0.07]


## 3. 第 3 步：RoPE，旋转位置编码

位置不再加到 $X$ 上，而是在注意力里把 query 和 key **旋转**。把每个头的 $d_h$ 个特征两两配对，第 $j$ 对看成平面上的一个点，位置 $p$ 的向量转过角度 $p\,\theta_j$：

$$
\theta_j = \text{base}^{-2j/d_h}, \qquad
\begin{pmatrix} x'_{2j} \\ x'_{2j+1} \end{pmatrix} = \begin{pmatrix} \cos p\theta_j & -\sin p\theta_j \\ \sin p\theta_j & \cos p\theta_j \end{pmatrix} \begin{pmatrix} x_{2j} \\ x_{2j+1} \end{pmatrix}
$$

把这个 2×2 矩阵记作 $R(p\theta_j)$。旋转不改变长度，而且 $R(a)^{\top} R(b) = R(b - a)$：先转 $b$ 再倒转 $a$，等于只转 $b - a$。所以位置 $p$ 的 query 和位置 $p'$ 的 key 各转一次之后，点积只取决于相对距离 $p' - p$：

$$
\big(R(p\theta)\, q\big)^{\top} \big(R(p'\theta)\, k\big) = q^{\top} R\big((p' - p)\,\theta\big)\, k
$$

- **`rope_angles`**：一次性算好每个位置、每一对的 $\cos$ 和 $\sin$，两张 `[max_seq_len, d_h/2]` 的表。
- **`apply_rope`**：把偶数位和奇数位取出来，做上面的 2×2 旋转，再交错拼回去。

这里 $d_h = 4$，只有 2 对，转速是 $1$ 和 $0.01$。

In [9]:
def rope_angles(head_dim: int, max_seq_len: int, base: float = 10000.0) -> tuple[Tensor, Tensor]:
    """cos and sin of p * theta_j for every position p and pair j: two tables of shape [max_seq_len, head_dim/2]."""
    theta = base ** (-torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)   # [head_dim/2]: 1, ..., small
    angle = torch.arange(max_seq_len, dtype=torch.float32)[:, None] * theta           # [max_seq_len, head_dim/2]
    return angle.cos(), angle.sin()


def apply_rope(x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
    """Turn each pair (x_2j, x_2j+1) of each row by its angle. x: [..., n, head_dim]; cos, sin: [n, head_dim/2]."""
    even, odd = x[..., 0::2], x[..., 1::2]
    return torch.stack((even * cos - odd * sin, even * sin + odd * cos), dim=-1).flatten(-2)

In [10]:
cos, sin = rope_angles(cfg.head_dim, cfg.max_seq_len, cfg.rope_base)
speeds = torch.atan2(sin[1], cos[1])                                   # angle at position 1 = theta_j, read from the table
print("cos, sin tables", list(cos.shape), "  speeds theta_j:", [round(t, 4) for t in speeds.tolist()])
assert torch.allclose(speeds, torch.tensor([1.0, 0.01]))

q_vec, k_vec = torch.randn(cfg.head_dim), torch.randn(cfg.head_dim)
def score(p, p_key):
    q = apply_rope(q_vec[None], cos[p:p + 1], sin[p:p + 1])[0]
    k = apply_rope(k_vec[None], cos[p_key:p_key + 1], sin[p_key:p_key + 1])[0]
    return (q @ k).item()

print(f"score(1, 4) = {score(1, 4):.4f}   score(9, 12) = {score(9, 12):.4f}   same distance 3, same score")
print(f"score(1, 5) = {score(1, 5):.4f}   distance 4, different score")
assert math.isclose(score(1, 4), score(9, 12), abs_tol=1e-4)
x = torch.randn(5, cfg.head_dim)
assert torch.allclose(apply_rope(x, cos[:5], sin[:5]).norm(dim=-1), x.norm(dim=-1), atol=1e-5)   # lengths unchanged

cos, sin tables [16, 2]   speeds theta_j: [1.0, 0.01]
score(1, 4) = -0.1046   score(9, 12) = -0.1046   same distance 3, same score
score(1, 5) = -0.0519   distance 4, different score


## 4. 第 4 步：GQA 因果自注意力

**输入** $X$ `[B, n, d]`，**输出** `[B, n, d]`。

$$
\begin{aligned}
Q &= X W^{Q} \ (h \text{ heads}), \qquad K = X W^{K},\ V = X W^{V} \ (h_{kv} \text{ heads}) \\
\text{head}_i &= \operatorname{softmax}\!\left(\frac{\operatorname{RoPE}(Q_i)\, \operatorname{RoPE}(K_{g(i)})^{\top}}{\sqrt{d_h}} + M^{\text{causal}}\right) V_{g(i)}, \qquad g(i) = \left\lfloor \frac{i}{h / h_{kv}} \right\rfloor \\
\operatorname{Attn}(X) &= [\,\text{head}_0 \Vert \cdots \Vert \text{head}_{h-1}\,]\, W^{O}
\end{aligned}
$$

- **GQA**：$W^{K}$、$W^{V}$ 只输出 $h_{kv}$ 个头。query 头 $i$ 读第 $g(i)$ 组的 $K$、$V$（头和组都从 0 编号）。这里 4 个 query 头、2 组 K/V：头 0、1 读第 0 组，头 2、3 读第 1 组。
- **为什么这么做**：生成时要把每层的 $K$、$V$ 存进 KV cache。组越少，cache 越小，每步要从显存读的字节越少。这里 cache 只有 MHA 的一半，LLaMA 3 8B 是四分之一。
- **RoPE** 只转 $Q$ 和 $K$，$V$ 不动。
- **因果 mask**：位置 $i$ 只能读 $\le i$ 的 key。带 cache 时，新 query 的位置要加上前面已有的长度，所以代码里用 `tril(diagonal=T - n)`。
- **KV cache 接口**：传入前面位置的 $(K, V)$，就把新算的接在后面，再一起返回。

In [11]:
class Attention(nn.Module):
    """Causal self-attention with RoPE and grouped-query attention; can read and extend a KV cache."""

    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.n_heads % cfg.n_kv_heads == 0, "query heads must split evenly into key/value groups"
        self.h, self.h_kv, self.d_h = cfg.n_heads, cfg.n_kv_heads, cfg.head_dim
        self.wq = nn.Linear(cfg.d_model, self.h * self.d_h, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.h_kv * self.d_h, bias=False)   # fewer K, V heads than Q heads
        self.wv = nn.Linear(cfg.d_model, self.h_kv * self.d_h, bias=False)
        self.wo = nn.Linear(self.h * self.d_h, cfg.d_model, bias=False)

    def forward(self, x: Tensor, cos: Tensor, sin: Tensor,
                cache: Optional[tuple[Tensor, Tensor]] = None) -> tuple[Tensor, tuple[Tensor, Tensor]]:
        # x [B, n, d] at the positions given by cos, sin [n, d_h/2]; cache: (K, V) of the earlier positions.
        B, n, _ = x.shape
        q = self.wq(x).view(B, n, self.h, self.d_h).transpose(1, 2)        # [B, h, n, d_h]
        k = self.wk(x).view(B, n, self.h_kv, self.d_h).transpose(1, 2)     # [B, h_kv, n, d_h]
        v = self.wv(x).view(B, n, self.h_kv, self.d_h).transpose(1, 2)     # [B, h_kv, n, d_h]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)            # positions enter here; V is not turned
        if cache is not None:
            k = torch.cat([cache[0], k], dim=2)                            # [B, h_kv, T, d_h], T = past + n
            v = torch.cat([cache[1], v], dim=2)
        cache = (k, v)
        group = self.h // self.h_kv
        k = k.repeat_interleave(group, dim=1)                              # [B, h, T, d_h]: K/V head g serves
        v = v.repeat_interleave(group, dim=1)                              # query heads g*group .. g*group+group-1
        T = k.shape[2]
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_h)            # [B, h, n, T]
        causal = torch.ones(n, T, dtype=torch.bool, device=x.device).tril(diagonal=T - n)  # query i reads keys <= its position
        weights = F.softmax(scores.masked_fill(~causal, float("-inf")), dim=-1)
        out = (weights @ v).transpose(1, 2).reshape(B, n, self.h * self.d_h)   # concatenate the heads
        return self.wo(out), cache

### 4b. 逐行拆开 `forward`

用一个模块，在 RMSNorm 之后的 $X$ 上一行一行地跑，每行看形状。

In [12]:
attn = Attention(cfg)
x = Xn                                                                 # the attention input is the normalized X
cos_n, sin_n = cos[:n], sin[:n]
q = attn.wq(x).view(B, n, attn.h, attn.d_h).transpose(1, 2)
k = attn.wk(x).view(B, n, attn.h_kv, attn.d_h).transpose(1, 2)
v = attn.wv(x).view(B, n, attn.h_kv, attn.d_h).transpose(1, 2)
print("q", list(q.shape), "  k", list(k.shape), "  v", list(v.shape), "  (4 query heads, 2 K/V heads)")
print("W_Q", list(attn.wq.weight.shape), "  W_K", list(attn.wk.weight.shape), "  W_V", list(attn.wv.weight.shape))

q, k = apply_rope(q, cos_n, sin_n), apply_rope(k, cos_n, sin_n)      # positions enter here
group = attn.h // attn.h_kv
k_rep, v_rep = k.repeat_interleave(group, dim=1), v.repeat_interleave(group, dim=1)
print("after repeat_interleave: k", list(k_rep.shape), "  query head -> K/V head:", [i // group for i in range(attn.h)])
assert torch.equal(k_rep[:, 0], k_rep[:, 1]) and torch.equal(k_rep[:, 1], k[:, 0])   # heads 0 and 1 share group 0

q [1, 4, 7, 4]   k [1, 2, 7, 4]   v [1, 2, 7, 4]   (4 query heads, 2 K/V heads)
W_Q [16, 16]   W_K [8, 16]   W_V [8, 16]
after repeat_interleave: k [1, 4, 7, 4]   query head -> K/V head: [0, 0, 1, 1]


In [13]:
scores = q @ k_rep.transpose(-2, -1) / math.sqrt(attn.d_h)             # [B, h, n, n]
causal = torch.ones(n, n, dtype=torch.bool).tril()
weights = F.softmax(scores.masked_fill(~causal, float("-inf")), dim=-1)
out = attn.wo((weights @ v_rep).transpose(1, 2).reshape(B, n, attn.h * attn.d_h))
print("scores", list(scores.shape), "  weights row sums:", [round(s, 3) for s in weights.sum(-1)[0, 0].tolist()])
print("weights of head 0 (zeros above the diagonal):")
print(weights[0, 0].detach().numpy().round(2))

module_out, (K_cache, V_cache) = attn(x, cos_n, sin_n)
assert torch.allclose(out, module_out, atol=1e-6)
reference = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=True)   # PyTorch's own kernel
assert torch.allclose((weights @ v_rep), reference, atol=1e-6)
print("module output", list(module_out.shape), "  cache K", list(K_cache.shape), "  cache V", list(V_cache.shape))

scores [1, 4, 7, 7]   weights row sums: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
weights of head 0 (zeros above the diagonal):
[[1.   0.   0.   0.   0.   0.   0.  ]
 [0.53 0.47 0.   0.   0.   0.   0.  ]
 [0.21 0.52 0.27 0.   0.   0.   0.  ]
 [0.3  0.24 0.25 0.21 0.   0.   0.  ]
 [0.29 0.12 0.19 0.21 0.19 0.   0.  ]
 [0.2  0.12 0.17 0.21 0.19 0.11 0.  ]
 [0.14 0.18 0.16 0.14 0.12 0.21 0.05]]
module output [1, 7, 16]   cache K [1, 2, 7, 4]   cache V [1, 2, 7, 4]


## 5. 第 5 步：SwiGLU 前馈网络

**输入**和**输出**都是 `[B, n, d]`，中间宽 $d_{ff}$。

$$
\operatorname{FFN}(x) = \big(\operatorname{SiLU}(x W_1) \odot x W_3\big)\, W_2, \qquad \operatorname{SiLU}(z) = z\,\sigma(z)
$$

- **门控**：$x$ 被投影两次，过 SiLU 的那一路乘到线性的那一路上，逐元素相乘，所以叫"门"。
- **三个矩阵**：$W_1$ 和 $W_3$ 是 $d \to d_{ff}$，$W_2$ 是 $d_{ff} \to d$。参数量 $3\,d\,d_{ff}$，这里是 $3 \cdot 16 \cdot 48 = 2304$。
- **为什么效果好**：在同样的参数量和计算量下，门控 FFN 的困惑度比普通 ReLU 或 GELU FFN 低（Shazeer 2020），所以 LLaMA、Qwen、Mistral 都用它。

In [14]:
class SwiGLU(nn.Module):
    """FFN(x) = (SiLU(x W1) * x W3) W2: the SiLU branch gates the linear branch, element by element."""

    def __init__(self, d: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff, bias=False)   # gate branch
        self.w3 = nn.Linear(d, d_ff, bias=False)   # linear ("up") branch
        self.w2 = nn.Linear(d_ff, d, bias=False)   # back to d ("down")

    def forward(self, x: Tensor) -> Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))   # [B, n, d] -> [B, n, d_ff] -> [B, n, d]

In [15]:
ffn = SwiGLU(cfg.d_model, cfg.d_ff)
gate, up = F.silu(ffn.w1(Xn)), ffn.w3(Xn)
print("x", list(Xn.shape), "-> gate, up", list(gate.shape), "-> FFN", list(ffn(Xn).shape))
assert torch.allclose(ffn(Xn), ffn.w2(gate * up), atol=1e-6)
print("parameters:", sum(p.numel() for p in ffn.parameters()), "= 3 * d * d_ff =", 3 * cfg.d_model * cfg.d_ff)

x [1, 7, 16] -> gate, up [1, 7, 48] -> FFN [1, 7, 16]
parameters: 2304 = 3 * d * d_ff = 2304


## 6. 第 6 步：Block = 两个 pre-norm 子层

$$
x \leftarrow x + \operatorname{Attn}\big(\operatorname{RMSNorm}(x)\big), \qquad x \leftarrow x + \operatorname{FFN}\big(\operatorname{RMSNorm}(x)\big)
$$

- **pre-norm**：归一化放在子层的**输入**处，残差流 $x$ 本身从不被归一化，只是不断加上各子层的输出。从输出到输入有一条恒等路径，梯度可以直接流回去，深层网络更好训练。
- **代价**：残差流的数值会随层数变大，所以整个模型最后要再加一个 RMSNorm（第 7 步）。

In [16]:
class Block(nn.Module):
    """Pre-norm: x <- x + Attention(RMSNorm(x)); x <- x + SwiGLU(RMSNorm(x))."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(self, x: Tensor, cos: Tensor, sin: Tensor,
                cache: Optional[tuple[Tensor, Tensor]] = None) -> tuple[Tensor, tuple[Tensor, Tensor]]:
        a, cache = self.attn(self.attn_norm(x), cos, sin, cache)
        x = x + a                                     # the residual stream itself is never normalized
        x = x + self.ffn(self.ffn_norm(x))
        return x, cache

In [17]:
block = Block(cfg)
Y, (K1, V1) = block(X, cos_n, sin_n)
print("X", list(X.shape), "-> Block -> Y", list(Y.shape))
a, _ = block.attn(block.attn_norm(X), cos_n, sin_n)
assert torch.allclose(Y, (X + a) + block.ffn(block.ffn_norm(X + a)), atol=1e-6)   # the two residual additions

X [1, 7, 16] -> Block -> Y [1, 7, 16]


## 7. 第 7 步：整个模型

$$
X_0 = E[\mathbf{t}], \qquad X_\ell = \operatorname{Block}_\ell(X_{\ell-1}),\ \ell = 1, \dots, L, \qquad Z = \operatorname{RMSNorm}(X_L)\, W^{\text{head}}
$$

**输入** ids `[B, n]`，**输出** logits $Z$ `[B, n, V]`，外加每层的 $(K, V)$。

- **RoPE 表**用 `register_buffer` 存，跟着模型移动设备，但不是参数，也不存进 checkpoint（`persistent=False`）。
- **`start`**：这一段 token 在整句里从第几个位置开始。生成时每次只送一个新 token，RoPE 要用它真正的位置，所以必须传进来。
- **LM head 默认不共用权重**，和 LLaMA 一样。把 `tie_embeddings` 设成 `True`，就和 GPT-2 一样用 embedding 表的转置。

In [18]:
class DecoderLM(nn.Module):
    """Token embedding -> L pre-norm blocks -> final RMSNorm -> LM head. No position vector is added."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layers))
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.embed.weight   # one [V, d] matrix for input and output
        cos, sin = rope_angles(cfg.head_dim, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, ids: Tensor, caches: Optional[list] = None, start: int = 0) -> tuple[Tensor, list]:
        """ids [B, n] at positions start .. start+n-1 -> logits [B, n, V] and every layer's (K, V)."""
        n = ids.shape[1]
        assert start + n <= self.cfg.max_seq_len, "longer than the RoPE tables"
        cos, sin = self.rope_cos[start:start + n], self.rope_sin[start:start + n]
        x = self.embed(ids)                                            # [B, n, d]
        new_caches = []
        for i, block in enumerate(self.blocks):
            x, cache = block(x, cos, sin, None if caches is None else caches[i])
            new_caches.append(cache)
        return self.lm_head(self.norm(x)), new_caches                  # [B, n, V]

In [19]:
torch.manual_seed(0)
model = DecoderLM(cfg)
logits, caches = model(ids)
print("ids", list(ids.shape), "-> logits", list(logits.shape))
print("caches: one (K, V) per layer;", len(caches), "layers, K", list(caches[0][0].shape))

d, dh = cfg.d_model, cfg.head_dim
parts = {
    "embedding E": cfg.vocab_size * d,
    "per block: 2 RMSNorm": 2 * d,
    "per block: W_Q, W_O": 2 * d * cfg.n_heads * dh,
    "per block: W_K, W_V": 2 * d * cfg.n_kv_heads * dh,
    "per block: SwiGLU": 3 * d * cfg.d_ff,
    "final RMSNorm": d,
    "LM head": d * cfg.vocab_size,
}
per_block = sum(v for k, v in parts.items() if k.startswith("per block"))
total = parts["embedding E"] + cfg.n_layers * per_block + parts["final RMSNorm"] + parts["LM head"]
for k, v in parts.items():
    print(f"  {k:<24} {v:>5}")
print(f"  total = {total}, counted in the model: {sum(p.numel() for p in model.parameters())}")
assert total == sum(p.numel() for p in model.parameters())

ids [1, 7] -> logits [1, 7, 11]
caches: one (K, V) per layer; 2 layers, K [1, 2, 7, 4]
  embedding E                176
  per block: 2 RMSNorm        32
  per block: W_Q, W_O        512
  per block: W_K, W_V        256
  per block: SwiGLU         2304
  final RMSNorm               16
  LM head                    176
  total = 6576, counted in the model: 6576


## 8. 第 8 步：loss，错开一格

$$
\mathcal{L} = -\frac{1}{n-1}\sum_{j=0}^{n-2} \log \operatorname{softmax}(Z_j)_{t_{j+1}}
$$

去掉 logits 的最后一行（它没有标签），去掉 ids 的第一个 token（它不是任何位置的答案），剩下的一一对应。因果 mask 保证位置 $j$ 只看到 $t_0, \dots, t_j$，所以一次前向就能训练全部 $n-1$ 个预测。随机初始化时，loss 和 $\log V = \log 11 \approx 2.40$ 在同一量级。

`pad_id`：一个 batch 里句子长短不一时，短句末尾补 `<pad>`；把它传进去，标签是 `<pad>` 的位置就不算 loss（`ignore_index`）。这里两句一样长，用不到。

In [20]:
def next_token_loss(logits: Tensor, ids: Tensor, pad_id: Optional[int] = None) -> Tensor:
    """Position j predicts token j+1: drop the last row of logits and the first token of ids."""
    return F.cross_entropy(logits[:, :-1].reshape(-1, logits.shape[-1]), ids[:, 1:].reshape(-1),
                           ignore_index=-100 if pad_id is None else pad_id)

In [21]:
loss = next_token_loss(logits, ids)
manual = -F.log_softmax(logits[0, :-1], dim=-1)[torch.arange(n - 1), ids[0, 1:]].mean()
assert torch.allclose(loss, manual)
print(f"loss = {loss.item():.4f}   log V = {math.log(cfg.vocab_size):.4f}")

loss = 2.7077   log V = 2.3979


**训练。** 两句话一起训练 300 步。loss 不会降到 0：`<bos>` 后面是 `we` 还是 `you`，各占一半，单凭 `<bos>` 无从判断。这一个位置最少也要付出 $\ln 2$，平均到每句的 6 个预测上，下限是 $\ln 2 / 6 \approx 0.1155$。其余 5 个位置都能完全学会。

In [22]:
def train(model: DecoderLM, steps: int = 300, lr: float = 1e-2) -> list:
    """Fit the two sentences; returns the loss every 50 steps."""
    batch = torch.cat([encode(s) for s in SENTENCES])                  # [2, 7]
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    history = []
    for step in range(steps + 1):
        logits, _ = model(batch)
        loss = next_token_loss(logits, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            history.append(round(loss.item(), 4))
    return history

In [23]:
history = train(model, steps=300)
print("loss every 50 steps:", history)
print(f"floor ln 2 / 6 = {math.log(2) / 6:.4f}")

loss every 50 steps: [2.6379, 0.1247, 0.1191, 0.1176, 0.1169, 0.1165, 0.1163]
floor ln 2 / 6 = 0.1155


## 9. 第 9 步：生成与 KV cache

- **prefill**：把 prompt 一次性送进去，算出所有位置的 logits，同时把每层的 $K$、$V$ 存进 cache。这一步一次处理很多 token，瓶颈是算力。
- **decode**：之后每步只送 1 个新 token，位置是 `start = 当前长度 - 1`。它的 query 去读 cache 里所有过去的 key，新的 $K$、$V$ 接到 cache 后面。这一步每次只算一个 token，算力用不满，瓶颈是显存带宽：每步都要把全部权重和整个 KV cache 从显存读一遍。batch 小、上下文短时主要是读权重；上下文越长、batch 越大，KV cache 占的比例越高。
- **cache 大小**：$2 \cdot L \cdot T \cdot h_{kv} \cdot d_h$ 个数（$K$ 和 $V$ 各一份）。GQA 让它缩小到 MHA 的 $h_{kv}/h$。以 LLaMA 3 8B 为例：每个 token 存 $2 \cdot 32 \cdot 8 \cdot 128 = 65{,}536$ 个数，bf16 下 128 KB；8K 上下文约 1 GB，换成 MHA 会是 4 GB（权重本身约 16 GB）。
- **批量生成**：一个 batch 里各句结束得有早有晚，已经生成过 `<eos>` 的句子之后只补 `<eos>`，全部结束才停。

In [24]:
@torch.no_grad()
def generate(model: DecoderLM, prompt: Tensor, max_new_tokens: int, temperature: float = 0.0,
             eos_id: Optional[int] = None) -> Tensor:
    """Prefill the prompt once, then feed one new token per step and reuse every layer's K and V."""
    assert temperature >= 0, "temperature 0 is greedy; a negative one would invert the distribution"
    logits, caches = model(prompt)                                     # prefill: [B, n, V]
    ids = prompt
    done = torch.zeros(prompt.shape[0], 1, dtype=torch.bool, device=prompt.device)   # rows that emitted eos
    steps = min(max_new_tokens, model.cfg.max_seq_len - prompt.shape[1])   # never run past the RoPE tables
    for step in range(steps):
        last = logits[:, -1]                                           # only the last position predicts
        if temperature == 0:
            next_id = last.argmax(dim=-1, keepdim=True)                # greedy
        else:
            next_id = torch.multinomial(F.softmax(last / temperature, dim=-1), 1)
        if eos_id is not None:
            next_id = next_id.masked_fill(done, eos_id)                # a finished row keeps emitting eos
            done = done | (next_id == eos_id)
        ids = torch.cat([ids, next_id], dim=1)
        if step == steps - 1 or bool(done.all()):
            break                                                      # no forward pass whose logits go unused
        logits, caches = model(next_id, caches, start=ids.shape[1] - 1)   # decode: [B, 1, V], at its own position
    return ids


def kv_cache_numbers(cfg: Config, tokens: int) -> int:
    """Numbers stored in the KV cache: 2 (K and V) x layers x tokens x h_kv x d_h."""
    return 2 * cfg.n_layers * tokens * cfg.n_kv_heads * cfg.head_dim

先手动走一遍：prefill 两个 token，再逐个 decode，看 cache 怎么变长。

In [25]:
model.eval()
with torch.no_grad():
    prompt = torch.tensor([[BOS, VOCAB.index("you")]])
    logits_p, caches = model(prompt)                                  # prefill
    print(f"prefill {list(prompt.shape)} -> logits {list(logits_p.shape)}, cache K {list(caches[0][0].shape)}")
    seq = prompt
    next_id = logits_p[:, -1].argmax(-1, keepdim=True)
    for _ in range(4):
        seq = torch.cat([seq, next_id], dim=1)
        logits_s, caches = model(next_id, caches, start=seq.shape[1] - 1)   # decode one token
        assert torch.allclose(logits_s[:, -1], model(seq)[0][:, -1], atol=1e-5)   # same logits as rerunning the whole prefix
        print(f"decode {VOCAB[next_id.item()]!r:<8} at position {seq.shape[1] - 1} -> cache K {list(caches[0][0].shape)}")
        next_id = logits_s[:, -1].argmax(-1, keepdim=True)
print("so far:", decode_ids(seq[0]))

prefill [1, 2] -> logits [1, 2, 11], cache K [1, 2, 2, 4]
decode 'read'   at position 2 -> cache K [1, 2, 3, 4]
decode 'a'      at position 3 -> cache K [1, 2, 4, 4]
decode 'short'  at position 4 -> cache K [1, 2, 5, 4]
decode 'book'   at position 5 -> cache K [1, 2, 6, 4]
so far: <bos> you read a short book


In [26]:
for first in ["we", "you"]:
    prompt = torch.tensor([[BOS, VOCAB.index(first)]])
    print(decode_ids(prompt[0]), "->", decode_ids(generate(model, prompt, 8, eos_id=EOS)[0]))

# generate() picks the same greedy tokens as rerunning the whole prefix (the cell above checks the logits).
prompt = torch.tensor([[BOS, VOCAB.index("we")]])
seq = prompt
for _ in range(5):
    seq = torch.cat([seq, model(seq)[0][:, -1].argmax(-1, keepdim=True)], dim=1)
assert torch.equal(generate(model, prompt, 5), seq)
print(f"KV cache for 7 tokens: {kv_cache_numbers(cfg, 7)} numbers with GQA, {kv_cache_numbers(Config(n_kv_heads=cfg.n_heads), 7)} with MHA")

<bos> we -> <bos> we watch a short video <eos>
<bos> you -> <bos> you read a short book <eos>
KV cache for 7 tokens: 224 numbers with GQA, 448 with MHA


## 最后：一次跑完 `main()`

`.py` 的 `main()` 把上面的流程串起来：重新设种子、建模型、打印形状和参数量、训练、从两个 prompt 生成，再比较 GQA 和 MHA 的 cache 大小。

In [27]:
def main() -> None:
    torch.manual_seed(0)
    cfg = Config()
    model = DecoderLM(cfg)
    ids = encode(SENTENCES[0])                                         # BOS we watch a short video EOS: [1, 7]
    logits, caches = model(ids)
    print("ids", list(ids.shape), "->", decode_ids(ids[0]))
    print("logits", list(logits.shape), "  K per layer", list(caches[0][0].shape), "  V per layer", list(caches[0][1].shape))
    print("parameters:", sum(p.numel() for p in model.parameters()))
    print("loss before training:", round(next_token_loss(logits, ids).item(), 4), " log V =", round(math.log(cfg.vocab_size), 4))
    print("loss every 50 steps:", train(model))
    for first in ["we", "you"]:
        prompt = torch.tensor([[BOS, VOCAB.index(first)]])
        print("generate from", decode_ids(prompt[0]), "->", decode_ids(generate(model, prompt, 8, eos_id=EOS)[0]))
    print("KV cache for 7 tokens:", kv_cache_numbers(cfg, 7), "numbers; with MHA it would be",
          kv_cache_numbers(Config(n_kv_heads=cfg.n_heads), 7))

In [28]:
main()

ids [1, 7] -> <bos> we watch a short video <eos>
logits [1, 7, 11]   K per layer [1, 2, 7, 4]   V per layer [1, 2, 7, 4]
parameters: 6576
loss before training: 2.7077  log V = 2.3979


loss every 50 steps: [2.6379, 0.1247, 0.1191, 0.1176, 0.1169, 0.1165, 0.1163]
generate from <bos> we -> <bos> we watch a short video <eos>
generate from <bos> you -> <bos> you read a short book <eos>
KV cache for 7 tokens: 224 numbers; with MHA it would be 448


## 形状总表

$B = 1$，$n = 7$，$d = 16$，$h = 4$，$h_{kv} = 2$，$d_h = 4$，$d_{ff} = 48$，$V = 11$。

| 步骤 | 输入 | 输出 |
|---|---|---|
| 1 · embedding | ids `[1, 7]` | $X$ `[1, 7, 16]` |
| 2 · RMSNorm | `[1, 7, 16]` | `[1, 7, 16]` |
| 4 · 投影出 $Q$ / $K$、$V$ | `[1, 7, 16]` | $Q$ `[1, 4, 7, 4]`，$K$、$V$ `[1, 2, 7, 4]` |
| 4 · 给 $Q$、$K$ 套 RoPE（第 3 步的函数） | $Q$、$K$ | 形状不变，只旋转 |
| 4 · 复制 K/V 到每个 query 头 | `[1, 2, 7, 4]` | `[1, 4, 7, 4]` |
| 4 · 分数和权重 | `[1, 4, 7, 4]` 和 `[1, 4, 4, 7]` | `[1, 4, 7, 7]` |
| 4 · 权重乘 $V$ | `[1, 4, 7, 7]` 和 `[1, 4, 7, 4]` | `[1, 4, 7, 4]` |
| 4 · 拼接各头，乘 $W^{O}$ | `[1, 4, 7, 4]` → `[1, 7, 16]` | `[1, 7, 16]` |
| 5 · SwiGLU | `[1, 7, 16]`，中间 `[1, 7, 48]` | `[1, 7, 16]` |
| 6 · Block × 2 | `[1, 7, 16]` | `[1, 7, 16]` |
| 7 · 最后的 RMSNorm 和 LM head | `[1, 7, 16]` | $Z$ `[1, 7, 11]` |
| 8 · loss | $Z$ 前 6 行，ids 后 6 个 | 标量 |
| 9 · 每步 decode | 新 token `[1, 1]` | $Z$ `[1, 1, 11]`，cache 多一行 |